# Check GPU Device

In [1]:
# Use the shell escape (!) to run the script on Colab's cloud filesystem
!python -c "import torch; print('CUDA Available:', torch.cuda.is_available()); print('GPU Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')"


CUDA Available: True
GPU Device: NVIDIA L4


# Mount Google Drive and Git Clone the repo

In [3]:
import os
from google.colab import drive

# 1. Mount Google Drive to access your processed dataset
print("Mounting Google Drive...")
drive.mount('/content/drive')

# 2. Define repository details (Using public HTTPS url)
repo_name = "SME_Credit_Risk"
repo_url = f"https://github.com/mirkosimunovic/{repo_name}.git"

# 3. Clone the public repo (or pull if it already exists)
if not os.path.exists(f"/content/{repo_name}"):
    print(f"\nCloning public repository: {repo_url}...")
    !git clone {repo_url}
else:
    print(f"\nRepository {repo_name} already exists. Pulling latest code changes...")
    %cd /content/{repo_name}
    !git pull
    %cd /content

# 4. Create the target directory inside the cloned repo
!mkdir -p /content/{repo_name}/data/raw

# 5. Copy the cleaned SME dataset from Google Drive
# NOTE: If you saved the CSV inside a specific folder in Google Drive, 
# adjust the source path below (e.g., "/content/drive/MyDrive/YourFolder/processed_sme_final.csv")
drive_source_path = "/content/drive/MyDrive/xAI_Banking_Paper/data/SBAnational.csv"
colab_target_path = f"/content/{repo_name}/data/raw/SBAnational.csv"

if os.path.exists(drive_source_path):
    !cp "{drive_source_path}" "{colab_target_path}"
    print("\n✓ Cleaned SBA dataset successfully copied from Google Drive to the cloned project")
else:
    print(f"\n⚠️ WARNING: Could not find your dataset at: {drive_source_path}")
    print("Please check your file path inside your Google Drive side panel and update 'drive_source_path'.")

# 6. Change active directory to your repository root
%cd /content/{repo_name}
print(f"\nActive directory set to: {os.getcwd()}")

Mounted at /content/drive

Cloning public repository: https://github.com/mirkosimunovic/SME_Credit_Risk.git...
Cloning into 'SME_Credit_Risk'...
remote: Enumerating objects: 145, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 145 (delta 48), reused 124 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (145/145), 18.45 MiB | 20.21 MiB/s, done.
Resolving deltas: 100% (48/48), done.

✓ Cleaned SBA dataset successfully copied from Google Drive to the cloned project
/content/SME_Credit_Risk

Active directory set to: /content/SME_Credit_Risk


# Sync the Google Drive files into my Colab session repo

In [9]:
import shutil
from pathlib import Path

drive_root = Path("/content/drive/MyDrive/xAI_Banking_Paper/SME_Credit_Risk")
colab_root = Path("/content/SME_Credit_Risk")

files = [
    "data/processed/X_train.csv",
    "data/processed/y_train.csv",
    "data/processed/X_oot.csv",
    "data/processed/y_oot.csv",
    "models/artifacts/imputer.joblib",
    "models/artifacts/scaler.joblib",
    "models/artifacts/xgboost_best.json",
    "models/artifacts/lightgbm_best.txt",
    "models/artifacts/catboost_best.bin"

]

if not drive_root.exists():
    raise FileNotFoundError(
        f"Drive folder not found: {drive_root}\n"
        "Mount Drive and check the folder name."
    )

for rel in files:
    source = drive_root / rel
    dest = colab_root / rel
    if not source.exists():
        print(f"SKIP (not on Drive): {rel}")
        continue
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, dest)
    mb = source.stat().st_size / 1e6
    print(f"copied {rel}  ({mb:.1f} MB)")

print("\nDone. Files are in the Colab repo.")

copied data/processed/X_train.csv  (69.9 MB)
copied data/processed/y_train.csv  (1.5 MB)
copied data/processed/X_oot.csv  (12.4 MB)
copied data/processed/y_oot.csv  (0.3 MB)
copied models/artifacts/imputer.joblib  (140.3 MB)
copied models/artifacts/scaler.joblib  (0.0 MB)
copied models/artifacts/xgboost_best.json  (2.1 MB)
copied models/artifacts/lightgbm_best.txt  (1.1 MB)
copied models/artifacts/catboost_best.bin  (0.3 MB)

Done. Files are in the Colab repo.


# Git Pull to sync

In [15]:
!git pull


remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 8 (delta 4), reused 8 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 3.44 KiB | 1.15 MiB/s, done.
From https://github.com/mirkosimunovic/SME_Credit_Risk
   49d5ed3..28c28ca  main       -> origin/main
Updating 49d5ed3..28c28ca
Fast-forward
 notebooks/run_commands.ipynb | 338 ++++++++++++++++++++++++++++++++-----------
 scripts/preprocess.py        |   7 +-
 2 files changed, 257 insertions(+), 88 deletions(-)


# Install requirements

In [5]:

# 1. Install heavy, CUDA-dependent system libraries first
!pip install "fknni[rapids12]" --extra-index-url=https://pypi.nvidia.com
!pip install faiss-gpu-cu12

# 2. Silently install the rest of our standard project requirements
!pip install -q -r requirements.txt


Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 179.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 257.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.5/114.5 kB 137.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.2/51.2 kB 80.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 GB 60.8 MB/s eta 0:00:00:00:0100:01
INFO: pip is looking at multiple versions of cugraph-cu12 to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 263.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 266.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 141.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 92.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.

# Check project tree

In [10]:
!find . -not -path './.git/*' -not -path './__pycache__/*' | sed -e 's|[^/]*/|│   |g' -e 's|│   \([^│]\)|├── \1|'

.
├── .gitignore
├── outputs
│   ├── figures
│   │   ├── roc_curves_comparison.png
│   ├── results
│   │   ├── metrics_benchmark.csv
├── .git
├── .cursorrules
├── notebooks
│   ├── SME_week1.ipynb
│   ├── run_commands.ipynb
├── data
│   ├── processed
│   │   ├── y_train.csv
│   │   ├── X_train.csv
│   │   ├── y_oot.csv
│   │   ├── .gitkeep
│   │   ├── X_oot.csv
│   ├── raw
│   │   ├── SBAnational.csv
├── scripts
│   ├── explainer.py
│   ├── trainer.py
│   ├── preprocess.py
├── models
│   ├── artifacts
│   │   ├── catboost_best.bin
│   │   ├── imputer.joblib
│   │   ├── xgboost_best.json
│   │   ├── scaler.joblib
│   │   ├── lightgbm_best.txt
│   │   ├── .gitkeep
├── requirements.txt


# Run pipeline scripts

In [16]:
!python scripts/preprocess.py

SBA SME preprocessing — chronological, leak-proof

[1] Load data/raw/SBAnational.csv
  Raw shape: (899164, 27)
  Dropped 1,997 rows with missing / unmapped MIS_Status.
  Paid in Full (0)=739,609 | Default (1)=157,558

[1b] Parse GrAppv and SBA_Appv currency strings -> float
  GrAppv: non-null=897,167  min=1000.0  max=5472000.0
  SBA_Appv: non-null=897,167  min=500.0  max=5472000.0

[2] Parse ApprovalDate, sort oldest -> newest, chronological split
  Rolled 4 two-digit years back by 100 years.
  Sorted span: 1966-05-18 -> 2014-06-25
  Train (oldest 85%): 1966-05-18 -> 2007-03-21  n=762,591
  OOT   (newest 15%): 2007-03-21 -> 2014-06-25  n=134,576
  Default rate  train=0.1532  OOT=0.3029

[3] NAICS_Sector + training-only target encoding (State, sector)
  State keys scored from train: 52
  Safest states (lowest train default rate):
State
MT    0.062095
WY    0.063191
VT    0.069453
SD    0.073834
ND    0.079167
  Riskiest states (highest train default rate):
State
GA    0.198766
IL    0.2

In [17]:
!python scripts/trainer.py

SME Credit Risk — two-step trainer + OOT evaluation
Project root: /content/SME_Credit_Risk

Creating output directories if missing:
  [ok] outputs/figures
  [ok] outputs/results
  [ok] models/artifacts

Loading chronological splits (NaNs still present):
  Loaded train: X=(762591, 22)  defaults=116,798  rate=0.153159
  Loaded oot: X=(134576, 22)  defaults=40,760  rate=0.302877

STEP 1  Stratified 5-fold CV on X_train / y_train
Imputer: FastKNNImputer.fit_transform on the train fold, then on stacked val.
         (Library has no fit/transform; val neighbors come from imputed train.)
Scaler:  StandardScaler on continuous columns (nunique > 10) — train fold only.
Imbalance: scale_pos_weight = n_neg / n_pos on the training fold.
OOT is held out of this entire loop.

Fold 1/5  n_train=610,072  n_val=152,519  scale_pos_weight=5.53
    FastKNNImputer.fit_transform on training slice (610,072 rows) ...
    FastKNNImputer.fit_transform on stacked reference+apply (762,591 rows) ...
    Continuous 

In [18]:
!python scripts/explainer.py

Week 3  SHAP explainability + Kendall's W stability (XGBoost champion)

[2] Loading frozen imputer / scaler and chronological splits
  Loading models/artifacts/imputer.joblib ...
  Imputer reference shape=(762591, 22)  encoded columns=22
  Loading models/artifacts/scaler.joblib ...
  Loading OOT features / targets ...
  Loaded oot: X=(134576, 22)  default rate=0.302877
  Transforming OOT (impute against frozen train reference, then scale) ...
    FastKNNImputer.fit_transform on stacked reference+apply (897,167 rows) ...
    Scaling 12 continuous columns: ['State', 'NAICS', 'Term', 'NoEmp', 'CreateJob', 'RetainedJob', 'RevLineCr', 'GrAppv', 'NAICS_Sector', 'Log_GrAppv', 'Guarantee_Ratio', 'Term_Years']
  Loading training features / targets ...
  Loaded train: X=(762591, 22)  default rate=0.153159
  Using imputer.reference_ as imputed X_train (champion training matrix) ...
    Scaling 12 continuous columns: ['State', 'NAICS', 'Term', 'NoEmp', 'CreateJob', 'RetainedJob', 'RevLineCr', 'GrA

# Sync the Colab session file to my Google Drive drive

In [19]:
import shutil
from pathlib import Path

src = Path("/content/SME_Credit_Risk")
dst = Path("/content/drive/MyDrive/xAI_Banking_Paper/SME_Credit_Risk")

files = [
    # preprocess.py (chronological splits — re-run on Colab)
    "data/processed/X_train.csv",
    "data/processed/y_train.csv",
    "data/processed/X_oot.csv",
    "data/processed/y_oot.csv",
    # trainer.py
    "models/artifacts/imputer.joblib",
    "models/artifacts/scaler.joblib",
    "models/artifacts/xgboost_best.json",
    "models/artifacts/lightgbm_best.txt",
    "models/artifacts/catboost_best.bin",
    "outputs/results/metrics_benchmark.csv",
    "outputs/figures/roc_curves_comparison.png",
    # explainer.py
    "outputs/figures/shap_summary.png",
    "outputs/figures/shap_importance.png",
    "outputs/results/stability_report.json"
]


if not dst.exists():
    raise FileNotFoundError(
        f"Drive folder not found: {dst}\n"
        "Re-run the drive.mount cell, then check the folder name."
    )

for rel in files:
    source = src / rel
    dest = dst / rel
    if not source.exists():
        print(f"SKIP (missing in Colab): {rel}")
        continue
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, dest)
    mb = source.stat().st_size / 1e6
    print(f"copied {rel}  ({mb:.1f} MB)")

print("\nDone. Confirm on Drive, then download that folder into your local SME_Credit_Risk repo.")

copied data/processed/X_train.csv  (68.2 MB)
copied data/processed/y_train.csv  (1.5 MB)
copied data/processed/X_oot.csv  (12.1 MB)
copied data/processed/y_oot.csv  (0.3 MB)
copied models/artifacts/imputer.joblib  (134.2 MB)
copied models/artifacts/scaler.joblib  (0.0 MB)
copied models/artifacts/xgboost_best.json  (2.1 MB)
copied models/artifacts/lightgbm_best.txt  (1.1 MB)
copied models/artifacts/catboost_best.bin  (0.3 MB)
copied outputs/results/metrics_benchmark.csv  (0.0 MB)
copied outputs/figures/roc_curves_comparison.png  (0.1 MB)
copied outputs/figures/shap_summary.png  (0.2 MB)
copied outputs/figures/shap_importance.png  (0.1 MB)
copied outputs/results/stability_report.json  (0.0 MB)

Done. Confirm on Drive, then download that folder into your local SME_Credit_Risk repo.


In [ ]:
!ls